# 03: Risk-Adjusted Forecasting with Monte Carlo

**Goal:** Combine GAO risk scores with IT Dashboard schedule data to generate risk-adjusted completion forecasts with confidence intervals.

**Method:** Monte Carlo simulation with 10,000 iterations per project

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Data

In [ ]:
projects = pd.read_csv('../data/projects_schedule_enhanced.csv')
agency_risk = pd.read_csv('../data/agency_risk_scores.csv', index_col=0)

# Map risk scores
risk_map = agency_risk['is_open'].to_dict()
projects['agency_risk_score'] = projects['agency_code'].map(risk_map).fillna(0.5)

print(f"Projects: {len(projects)}")
print(f"Mean risk score: {projects['agency_risk_score'].mean():.3f}")

## Monte Carlo Simulation

In [ ]:
def monte_carlo_forecast(row, risk_score, n_simulations=10000):
    np.random.seed(42)
    
    base_delay = max(0, row.get('schedule_variance_days', 0))
    spi = row.get('spi_estimate', 0.95)
    
    # Health penalty
    health = str(row.get('project_health', '')).lower()
    penalty = 60 if 'red' in health else (30 if 'yellow' in health else 0)
    
    # Risk multiplier
    multiplier = 1.0 + (risk_score * 0.5)
    
    # Simulate
    delay = np.random.normal(loc=base_delay + penalty, scale=20, size=n_simulations)
    delay = np.maximum(delay, 0)
    
    if spi < 1.0:
        extension = np.random.normal(loc=(1/spi - 1) * 90, scale=30, size=n_simulations)
        extension = np.maximum(extension, 0)
    else:
        extension = np.zeros(n_simulations)
    
    total = (delay + extension) * multiplier
    
    return {
        'p50': np.percentile(total, 50),
        'p80': np.percentile(total, 80),
        'p95': np.percentile(total, 95),
        'mean': np.mean(total),
        'std': np.std(total)
    }

In [ ]:
# Run for all projects
forecasts = []
for idx, row in projects.iterrows():
    if idx % 50 == 0:
        print(f"Processed {idx}/{len(projects)}...")
    
    fc = monte_carlo_forecast(row, row['agency_risk_score'])
    fc['project_id'] = row.get('project_id', f'proj_{idx}')
    fc['agency'] = row.get('agency_code', 'Unknown')
    fc['project_health'] = row.get('project_health', 'Unknown')
    forecasts.append(fc)

results = pd.DataFrame(forecasts)
results['risk_level'] = pd.cut(
    results['p80'],
    bins=[0, 30, 90, 180, 9999],
    labels=['Low', 'Medium', 'High', 'Critical']
)

print(f"\nCompleted {len(results)} forecasts")

## Results Visualization

In [ ]:
# Risk distribution
risk_dist = results['risk_level'].value_counts()
colors = {'Low': '#2ecc71', 'Medium': '#f1c40f', 'High': '#e67e22', 'Critical': '#e74c3c'}

plt.figure(figsize=(10, 6))
bars = plt.bar(risk_dist.index, risk_dist.values, color=[colors.get(x, '#3498db') for x in risk_dist.index])
plt.title('Project Risk Distribution')
plt.ylabel('Number of Projects')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}\n({height/len(results):.1%})',
             ha='center', va='bottom')
plt.show()

In [ ]:
# P80 delay histogram
plt.figure(figsize=(12, 6))
for level in ['Low', 'Medium', 'High', 'Critical']:
    subset = results[results['risk_level'] == level]
    if len(subset) > 0:
        plt.hist(subset['p80'], bins=20, alpha=0.6, label=level, color=colors.get(level))
plt.title('P80 Delay Distribution by Risk Level')
plt.xlabel('Days')
plt.ylabel('Count')
plt.legend()
plt.show()

In [ ]:
# Top 10 at-risk
top10 = results.nlargest(10, 'p80')[['project_id', 'agency', 'project_health', 'p50', 'p80', 'p95', 'risk_level']]
top10

In [ ]:
# Save results
results.to_csv('../data/hybrid_forecast_results.csv', index=False)
print("Results saved to hybrid_forecast_results.csv")